In [1]:
# ===========================================================================
# UA-SPEECH MODEL TRAINING - interactive driver
#
# All training logic lives in src/training/; this notebook only calls it and
# stores results, matching notebooks/01_data_pipeline.ipynb's convention -
# functions and architecture belong in src/, only the act of running training
# and storing models happens here.
#
#   src/training/models.py     model factory: acoustic / deep_frozen / deep_lora / fusion
#   src/training/runner.py     TrainingConfig, run_training() - the fold loop
#   src/training/baseline.py   Phase 2: frozen wav2vec + linear SVM baseline
#   src/training/engine.py     one epoch: AMP, gradient clipping, optimizer
#   src/training/metrics.py    accuracy / precision / recall / specificity / F1 / AUROC
#   src/training/reporting.py  predictions / metrics / confusion-matrix / ROC / embeddings I/O
#
# See ROADMAP.md for the phase plan this notebook implements (Phase 1 sanity
# check, Phase 2 baseline reproduction and comparison).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("UA-Speech Training Notebook")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))
print_kv("Speakers", df_m6["Speaker_ID"].nunique())


══════════════════════════════════════════════════════════════════════════════
  UA-SPEECH TRAINING NOTEBOOK
══════════════════════════════════════════════════════════════════════════════
  Manifest ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv
  Utterances .............................. 21381
  Speakers ................................ 28


In [2]:
# STAGE 1 - Pipeline sanity check ("smoke test"). Trains the cheapest model
# (MFCC-only) for one fold, one epoch, on a tiny slice of data. This is NOT a
# real result - it exists to confirm the whole chain (model init, optimizer,
# scheduler, AMP, gradient clipping, early stopping, checkpointing,
# TensorBoard logging, prediction/metric/confusion-matrix/ROC/embedding
# writers) actually runs end to end before spending GPU time on a real run.
from src.training.runner import TrainingConfig, run_training

smoke_cfg = TrainingConfig(
    task="detection", model="acoustic",
    epochs=1, max_folds=1, limit_samples=24,
    run_name="_smoke_test",
)
run_training(df_m6, smoke_cfg)


╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│               Model A — MFCC 1D-CNN (cepstral features only)               │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ................................... acoustic — Model A — MFCC 1D-CNN (cepstral features only)
  Cross-validation protocol ............... Leave-One-Speaker-Out
  Run name ................................ _smoke_test
  Device .................................. cuda
  Epochs / batch size ..................... 1 / 8
  LR (head / wav2vec backbone) ............ 0.001 / 0.0001
  Early stopping patience ................. 5 epochs on validation loss

─── Front end — short-time analysis ──────────────────────────────────────────
  Sam


──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: CF02  │  train 24 / val 6 / test 24
──────────────────────────────────────────────────────────────────────────────

─── Architecture — acoustic ──────────────────────────────────────────────────
    acoustic_pathway ......................     103,552 /     103,552 trainable (100.0%)
    classifier ............................       8,386 /       8,386 trainable (100.0%)
  TOTAL trainable ......................... 111,938 / 111,938 (100.00%)



  epoch 1/1 train                    0%|                                   | 0/3 [00:00<?, ?batch/s]

  epoch 1/1 train                   33%|█████████                  | 1/3 [00:01<00:02,  1.16s/batch]

  epoch 1/1 train                   67%|██████████████████         | 2/3 [00:01<00:00,  1.68batch/s]

...

  held-out test (CF02)              33%|█████████                  | 1/3 [00:00<00:00,  6.26batch/s]

  held-out test (CF02)              67%|██████████████████         | 2/3 [00:00<00:00,  5.80batch/s]

  held-out test (CF02)             100%|███████████████████████████| 3/3 [00:00<00:00,  5.80batch/s]

  Fold CF02 held-out test ................. accuracy=1.000, precision=0.000, recall=0.000, specificity=1.000, f1=0.000, auroc=nan

══════════════════════════════════════════════════════════════════════════════
  RESULTS — _SMOKE_TEST  (1 FOLD(S), 24 HELD-OUT UTTERANCES)
══════════════════════════════════════════════════════════════════════════════

─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean  std
    test_loss 0.6570  NaN
     accuracy 1.0000  NaN
    precision 0.0000  NaN
       recall 0.0000  NaN
  specificity 1.0000  NaN
           f1 0.0000  NaN
        auroc    NaN  NaN

  • Every LOSO fold holds out ONE speaker, who is entirely one class, so per-fold precision / recall / specificity / AUROC above are
  • degenerate — only 'accuracy' is meaningful per fold. The pooled numbers below are the ones comparable to the base paper.

─── Pooled across all folds (base-paper-style LOSO reporting) ────────────────
  accuracy       ██████

(                 mean  std
 test_loss    0.657023  NaN
 accuracy     1.000000  NaN
 precision    0.000000  NaN
 recall       0.000000  NaN
 specificity  1.000000  NaN
 f1           0.000000  NaN
 auroc             NaN  NaN,
 {'accuracy': 1.0,
  'precision': 0.0,
  'recall': 0.0,
  'specificity': 1.0,
  'f1': 0.0,
  'auroc': nan})

In [3]:
# STAGE 2 - Phase 2, step 1: reproduce the ICASSP base paper's feature
# extractor. Frozen wav2vec 2.0 (no LoRA, no fine-tuning) -> one 768-dim
# embedding per utterance PER HIDDEN-STATE LAYER (13 layers: the CNN
# feature-extractor output + 12 transformer layers). The base paper finds
# different layers win for different tasks (layer 1 for detection, layer 13
# for severity), so all 13 are extracted here rather than just the final
# layer - Stage 3 sweeps them to find which wins on this reproduction.
# Identical across every LOSO fold, so extracted once and cached to
# outputs/embeddings/.
from src.training.baseline import extract_frozen_embeddings_all_layers

frozen_embeddings_all_layers = extract_frozen_embeddings_all_layers(df_m6, batch_size=16)
print(f"Frozen embeddings (all layers): {frozen_embeddings_all_layers.shape}")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



══════════════════════════════════════════════════════════════════════════════
  EXTRACTING FROZEN WAV2VEC 2.0 EMBEDDINGS (ALL 13 LAYERS)
══════════════════════════════════════════════════════════════════════════════
  Utterances .............................. 21381
  Device .................................. cuda


  Frozen wav2vec 2.0 forward (13 layers)   0%|                          | 0/1337 [00:00<?]

  Frozen per-layer embeddings ............. extracted and cached to C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\embeddings\frozen_wav2vec_all_layers.npz
Frozen embeddings (all layers): (21381, 13, 768)


In [4]:
# STAGE 3 - Phase 2, step 2: frozen wav2vec 2.0 -> linear SVM, swept across
# all 13 layers and evaluated on the full 28-fold LOSO detection protocol -
# exactly the base paper's pipeline and per-layer comparison. The paper's
# own reported result is layer 1 at 93.95% accuracy; the best layer found
# here is the number every other model in this project has to beat to be a
# genuine improvement, not an assumed one. If the best layer or accuracy
# lands far from the paper's, that's worth investigating (preprocessing,
# VAD, clip length) before trusting the ablation table in Stage 4.
from src.training.baseline import sweep_svm_baseline_layers

detection_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="detection", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

best_layer = int(detection_layer_sweep.iloc[0]["layer"])
baseline_pooled = detection_layer_sweep.iloc[0].to_dict()
baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Detection")
print_kv("Best layer", f"{best_layer} (paper reports layer 1 at 93.95% accuracy)")
print_kv("Accuracy", f"{baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{baseline_pooled['f1']:.4f}")
print_kv("Recall (sensitivity)", f"{baseline_pooled['recall']:.4f}")
print_kv("Precision", f"{baseline_pooled['precision']:.4f}")
print_kv("Specificity", f"{baseline_pooled['specificity']:.4f}")
print_kv("AUROC", f"{baseline_pooled['auroc']:.4f}")


─── Layer 0 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run name ................................ baseline_svm_detection_layer0


  Fitting SVM per LOSO fold          0%|                                  | 0/28 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.9046 0.1782
    precision 0.5357 0.5079
       recall 0.4637 0.4697
  specificity 0.4409 0.4849
           f1 0.4860 0.4787
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ██████████████████░░   0.9044 →
  precision      ███████████████████░   0.9518  
  recall         █████████████████░░░   0.8652  
  specificity    ███████████████████░   0.9496  
  f1             ██████████████████░░   0.9064 →
  auroc          ███████████████████░   0.9615  

─── Layer 1 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run

  Fitting SVM per LOSO fold          0%|                                  | 0/28 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.9162 0.1463
    precision 0.5357 0.5079
       recall 0.4758 0.4699
  specificity 0.4404 0.4850
           f1 0.4981 0.4803
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ██████████████████░░   0.9161 →
  precision      ███████████████████░   0.9521  
  recall         ██████████████████░░   0.8879  
  specificity    ███████████████████░   0.9486  
  f1             ██████████████████░░   0.9189 →
  auroc          ███████████████████░   0.9735  

─── Layer 2 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run

...


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.8599 0.1865
    precision 0.5357 0.5079
       recall 0.4377 0.4504
  specificity 0.4221 0.4639
           f1 0.4685 0.4675
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       █████████████████░░░   0.8597 →
  precision      ██████████████████░░   0.9118  
  recall         ████████████████░░░░   0.8167  
  specificity    ██████████████████░░   0.9092  
  f1             █████████████████░░░   0.8617 →
  auroc          ██████████████████░░   0.9210  

─── Layer 11 / 12 ────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run

  Fitting SVM per LOSO fold          0%|                                  | 0/28 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.8356 0.2024
    precision 0.5357 0.5079
       recall 0.4243 0.4456
  specificity 0.4113 0.4520
           f1 0.4571 0.4626
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       █████████████████░░░   0.8354 →
  precision      ██████████████████░░   0.8885  
  recall         ████████████████░░░░   0.7916  
  specificity    ██████████████████░░   0.8858  
  f1             █████████████████░░░   0.8373 →
  auroc          ██████████████████░░   0.8979  

─── Layer 12 / 12 ────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run

  Fitting SVM per LOSO fold          0%|                                  | 0/28 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.8021 0.1994
    precision 0.5357 0.5079
       recall 0.4134 0.4358
  specificity 0.3887 0.4280
           f1 0.4504 0.4558
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ████████████████░░░░   0.8020 →
  precision      █████████████████░░░   0.8450  
  recall         ███████████████░░░░░   0.7712  
  specificity    █████████████████░░░   0.8373  
  f1             ████████████████░░░░   0.8064 →
  auroc          █████████████████░░░   0.8649  

─── Layer sweep summary (detection) — best layer first ───────────────────────
   layer  accuracy  precision  recall  specificity     f1  auroc
       1    0.9161     0.9521  0.8879       0.9486 0.9189 0.9735
       2    0.9122     0.9527  0.8795       0.9498 0.9147 0.9717
       0    0.9044     0.9518  0.8652       0.9496 0.9064 0.9615
       3  

In [5]:
# STAGE 4 - Phase 2, step 3: the same frozen wav2vec 2.0 + linear SVM layer
# sweep, on the severity task's 81-fold balanced leave-one-per-class-out
# protocol (config.DROPPED_FOR_BALANCE, corrected to match the base paper's
# stated exclusion criterion - see src/config.py). The paper's own reported
# result is layer 13 (final) at 44.56% accuracy (4-class) - a low absolute
# number, expected for a 4-way severity task, but the target to compare
# against before trusting the severity ablation in Stage 7.
from src.training.baseline import sweep_svm_baseline_layers

severity_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="severity", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

severity_best_layer = int(severity_layer_sweep.iloc[0]["layer"])
severity_baseline_pooled = severity_layer_sweep.iloc[0].to_dict()
severity_baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Severity")
print_kv("Best layer", f"{severity_best_layer} (paper reports layer 13/final at 44.56% accuracy)")
print_kv("Accuracy", f"{severity_baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{severity_baseline_pooled['f1']:.4f}")


─── Layer 0 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... severity
  Run name ................................ baseline_svm_severity_layer0

══════════════════════════════════════════════════════════════════════════════
  SEVERITY SPLITS (BALANCED LEAVE-ONE-PER-CLASS-OUT)
══════════════════════════════════════════════════════════════════════════════
  Speakers dropped for balance ............ ['M12', 'M08', 'M09']

─── Speakers per severity class ──────────────────────────────────────────────
  High .................................... F05, M10, M14
  Low ..................................... F02, M07, M16
  Mid ..................................... F04, M05, M11
  Very Low ................................ F03, M0

  Fitting SVM per LOSO fold          0%|                                  | 0/81 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.3406 0.0739
    precision 0.3403 0.0805
       recall 0.3409 0.0741
  specificity 0.7803 0.0246
           f1 0.3107 0.0627
        auroc 0.5968 0.0646

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ███████░░░░░░░░░░░░░   0.3406 →
  precision      ██████░░░░░░░░░░░░░░   0.3190  
  recall         ███████░░░░░░░░░░░░░   0.3406  
  specificity    ████████████████░░░░   0.7803  
  f1             ██████░░░░░░░░░░░░░░   0.3237 →
  auroc          ████████████░░░░░░░░   0.5927  

─── Layer 1 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... severity
  Run 

  Fitting SVM per LOSO fold          0%|                                  | 0/81 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.3371 0.0769
    precision 0.3298 0.0824
       recall 0.3373 0.0774
  specificity 0.7792 0.0257
           f1 0.3109 0.0678
        auroc 0.5726 0.0686

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ███████░░░░░░░░░░░░░   0.3371 →
  precision      ██████░░░░░░░░░░░░░░   0.3113  
  recall         ███████░░░░░░░░░░░░░   0.3370  
  specificity    ████████████████░░░░   0.7791  
  f1             ██████░░░░░░░░░░░░░░   0.3180 →
  auroc          ███████████░░░░░░░░░   0.5706  

─── Layer 2 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... severity
  Run 

...


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.4388 0.0543
    precision 0.4221 0.0521
       recall 0.4390 0.0551
  specificity 0.8130 0.0181
           f1 0.4178 0.0480
        auroc 0.6780 0.0483

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       █████████░░░░░░░░░░░   0.4387 →
  precision      ████████░░░░░░░░░░░░   0.4117  
  recall         █████████░░░░░░░░░░░   0.4386  
  specificity    ████████████████░░░░   0.8129  
  f1             ████████░░░░░░░░░░░░   0.4229 →
  auroc          █████████████░░░░░░░   0.6724  

─── Layer 11 / 12 ────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... severity
  Run 

  Fitting SVM per LOSO fold          0%|                                  | 0/81 [00:00<?]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.4386 0.0518
    precision 0.4220 0.0450
       recall 0.4390 0.0525
  specificity 0.8130 0.0173
           f1 0.4191 0.0426
        auroc 0.6808 0.0433

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       █████████░░░░░░░░░░░   0.4386 →
  precision      ████████░░░░░░░░░░░░   0.4141  
  recall         █████████░░░░░░░░░░░   0.4385  
  specificity    ████████████████░░░░   0.8129  
  f1             ████████░░░░░░░░░░░░   0.4245 →
  auroc          ██████████████░░░░░░   0.6751  

─── Layer 12 / 12 ────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... severity
  Run 

  Fitting SVM per LOSO fold          0%|                                  | 0/81 [00:00<?]

KeyboardInterrupt: 

In [ ]:
# STAGE 5 - Phase 2 step 3 + Phase 3 ablation: full-scale training of all six
# variants - Frozen wav2vec + MLP, LoRA wav2vec + MLP, MFCC CNN, concatenated
# Fusion, and the two Phase 6 attention-fusion variants - on the complete
# 28-fold LOSO detection protocol, full epochs, no sample caps. This is the
# comparison the paper's contribution rests on: fusion -> attention_fusion ->
# attention_fusion_praat, same two pathways/data, only the fusion mechanism
# changes.
#
# NOTE ON SCALE: this is the real run, not a demo - a full 28-fold LOSO pass
# of a wav2vec-fine-tuning variant is hours of GPU time, not minutes, so all
# six variants is realistically a long unattended job. On a laptop GPU that
# is not safe to run continuously for hours/days unattended, so this cell
# is session-bounded: SESSION_BUDGET_HOURS caps how long *this call* trains
# for, and run_training() stops cleanly at the next fold boundary once the
# budget is used up. run_training() also skips any fold whose output already
# exists on disk (see src/training/runner.py), so simply re-running this
# cell later resumes from wherever the last session stopped - across folds
# AND across variants - instead of restarting.
from src.training.runner import TrainingConfig, run_training
from src.training.models import MODEL_NAMES
from src.console import print_note
import time

SESSION_BUDGET_HOURS = 2.5  # lower for a supervised first session, raise once you know real fold timing
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

comparison_pooled = {"baseline_svm": baseline_pooled}

for model_name in MODEL_NAMES:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        comparison_pooled[model_name] = pooled

In [ ]:
# STAGE 6 - Phase 2/3 comparison table: baseline SVM vs. every trained variant,
# pooled metrics side by side. Saved to outputs/metrics/phase2_comparison.csv
# for the paper/report.
comparison_df = pd.DataFrame(comparison_pooled).T
comparison_df.index.name = "model"

comparison_path = config.METRICS_DIR / "phase2_comparison.csv"
comparison_df.to_csv(comparison_path)

print_header("Phase 2/3 Comparison - Detection")
print_kv("Saved to", comparison_path)
comparison_df

In [ ]:
# STAGE 6.5 - Pick which variants go on to the severity task. Stage 7 below
# runs the 81-fold severity protocol (leave-one-speaker-per-class-out), which
# is ~3x the fold count of detection's 28-fold LOSO - across all six variants
# that's the single largest chunk of total GPU time in this notebook. Only
# the architectures that actually won the detection comparison matter for
# the severity story, so rank Stage 5's results by F1 and carry forward just
# the top TOP_K_FOR_SEVERITY into Stage 7 instead of all six.
TOP_K_FOR_SEVERITY = 2

ranked_variants = sorted(
    (m for m in MODEL_NAMES if m in comparison_pooled),
    key=lambda m: comparison_pooled[m]["f1"], reverse=True,
)
severity_model_names = ranked_variants[:TOP_K_FOR_SEVERITY]

print_note(f"Running severity (81-fold) only for top {TOP_K_FOR_SEVERITY} "
          f"detection variants by F1: {severity_model_names}")

In [ ]:
# STAGE 7 - Severity task: the top variants from Stage 6.5 (by default, the
# best TOP_K_FOR_SEVERITY of the six by detection F1 - see the cell above) on
# the balanced leave-one-speaker-per-class-out protocol (81 folds; see
# src/splits.py and config.DROPPED_FOR_BALANCE), plus the Stage 4 severity
# SVM baseline for a like-for-like comparison table (mirrors Stage 5/6's
# detection pattern).
#
# Lower priority than Stage 5's detection run (the paper's primary
# comparison) - run this once Stage 5 is done or far enough along, since
# 81 folds is a larger job than 28 even before multiplying by variant count.
# Same session-bounded pattern as Stage 5: SESSION_BUDGET_HOURS caps this
# call, run_training() stops cleanly at a fold boundary, and re-running the
# cell resumes rather than restarts.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

SESSION_BUDGET_HOURS = 2.5  # lower for a supervised first session, raise once you know real fold timing
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

severity_pooled = {"baseline_svm": severity_baseline_pooled}

for model_name in severity_model_names:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

severity_df = pd.DataFrame(severity_pooled).T
severity_df.index.name = "model"

severity_path = config.METRICS_DIR / "phase3_severity_comparison.csv"
severity_df.to_csv(severity_path)

print_header("Phase 3 Comparison - Severity")
print_kv("Saved to", severity_path)
severity_df